# Deploy: AgentCore Runtime with Two Memories (S3 + AgentCore Memory)

This notebook deploys `agentcore_production.py` to **AgentCore Runtime** and provisions the two backends it uses:

1. **An S3 bucket** — where `ContextOffloader(S3Storage(...))` stores large tool outputs (the *context / data memory*, recalled by **exact reference**).
2. **An AgentCore Memory resource** — where the conversation is persisted (the *conversation memory*, recalled by **semantic similarity**, `STM_AND_LTM`).

Every provisioning step is **idempotent**: existing resources are reused, missing ones are created.

> This demo uses Strands Agents on Amazon Bedrock AgentCore. The two-memory split (semantic conversation memory + exact-reference data memory) is a general agent design and carries over to other frameworks.

## Who writes to S3 — and why the role needs permissions

The agent runs **inside AgentCore Runtime**, so the **Runtime execution role** is the principal that reads and writes the S3 bucket. That is why this notebook grants the role S3 access. The AgentCore *Memory* service is separate — it does not touch your bucket.

| Memory | Backend | Principal that accesses it |
|--------|---------|----------------------------|
| Conversation | AgentCore **Memory** service | Managed by AgentCore (via the session manager) |
| Context data | **S3** bucket | The **Runtime execution role** |

## Prerequisites

- AWS credentials with permissions for S3, IAM, Amazon Bedrock (model access to Claude), and `bedrock-agentcore` / `bedrock-agentcore-control`.
- `uv pip install 'bedrock-agentcore[strands-agents]' bedrock-agentcore-starter-toolkit boto3`
- Docker running locally (the starter toolkit builds a container).

## Before you start: AWS account and credentials

This notebook creates real AWS resources (S3, IAM role, AgentCore Memory + Runtime). You need an AWS account and credentials configured locally.

**1. Create an AWS account** (skip if you have one): [aws.amazon.com/free](https://aws.amazon.com/free). The free tier covers this demo — storage is a few KB.

**2. Create access keys:** AWS Console → **IAM → Users → your user → Security credentials → Create access key** → **Command Line Interface (CLI)**.

**3. Configure them locally** (install the [AWS CLI](https://docs.aws.amazon.com/cli/latest/userguide/getting-started-install.html) first):

```bash
aws configure                      # or: aws configure --profile my-profile
# AWS Access Key ID:     <your-access-key-id>
# AWS Secret Access Key: <your-secret-access-key>
# Default region name:   us-east-1
```

Using a named profile? Start Jupyter with `AWS_PROFILE=my-profile jupyter lab`. `boto3` resolves credentials from the standard credential chain — no keys in code.

**4. Enable Bedrock model access** once in the [Amazon Bedrock console](https://console.aws.amazon.com/bedrock/) → **Model access** → request Anthropic Claude (usually instant). The deployed agent uses Bedrock, so no API key lives in the container.

**5. Install dependencies:**

```bash
uv pip install 'bedrock-agentcore[strands-agents]' bedrock-agentcore-starter-toolkit boto3
```

> **No Docker needed locally.** The starter toolkit builds the agent in the cloud with AWS CodeBuild (ARM64) and deploys it — you don't manage containers or ECR. Docker is only required if you explicitly opt into `launch(local=True)` (run locally) or `launch(local_build=True)` (build locally).

In [ ]:
%pip install -r agent_requirements.txt bedrock-agentcore-starter-toolkit

# Verify versions: AgentCore Memory + Runtime APIs and the native ContextOffloader
import importlib.metadata as _m

def _ge(pkg, major, minor):
    v = _m.version(pkg)
    ok = tuple(int(x) for x in v.split(".")[:2]) >= (major, minor)
    assert ok, f"{pkg} {v} is too old (need >= {major}.{minor}). Re-run install + restart the kernel."
    return v

print("strands-agents             ", _ge("strands-agents", 1, 44), "— OK")
print("bedrock-agentcore          ", _ge("bedrock-agentcore", 1, 14), "— OK")
print("bedrock-agentcore-starter-toolkit", _ge("bedrock-agentcore-starter-toolkit", 0, 3), "— OK")

## Configuration

Edit `REGION` and `CONTEXT_BUCKET` for your account. The bucket name must be globally unique.

In [ ]:
import boto3, json, time
from botocore.exceptions import ClientError

REGION = 'us-east-1'
CONTEXT_BUCKET = 'my-agent-context-bucket'  # change to a globally-unique name you own
CONTEXT_PREFIX = 'log-artifacts/'
MEMORY_NAME = 'IncidentAgentMemory'
AGENT_NAME = 'LogAnalysisAgentTwoMemories'
ROLE_NAME = 'LogAnalysisAgentCoreExecutionRole'

session = boto3.Session(region_name=REGION)
account_id = session.client('sts').get_caller_identity()['Account']
iam = session.client('iam')
s3 = session.client('s3')

print(f'Account: {account_id}  |  Region: {REGION}')
print(f'Bucket:  {CONTEXT_BUCKET}/{CONTEXT_PREFIX}')
print(f'Memory:  {MEMORY_NAME}   Agent: {AGENT_NAME}')

## Step 1 — S3 bucket (create if missing, reuse if it exists)

`head_bucket` tells us whether the bucket already exists and is accessible. If not, we create it — handling the `us-east-1` quirk where `create_bucket` must **not** be given a `LocationConstraint`. Offloaded data is private, so we block public access.

In [ ]:
def ensure_bucket(bucket: str, region: str) -> str:
    """Create the bucket if it does not exist; reuse it if it does."""
    try:
        s3.head_bucket(Bucket=bucket)
        print(f"\u2705 Bucket '{bucket}' already exists \u2014 reusing it.")
        return bucket
    except ClientError as e:
        code = e.response['Error']['Code']
        if code == '403':
            raise RuntimeError(f"Bucket '{bucket}' is owned by another account. Pick another name.") from e
        if code not in ('404', 'NoSuchBucket'):
            raise

    if region == 'us-east-1':
        s3.create_bucket(Bucket=bucket)
    else:
        s3.create_bucket(Bucket=bucket, CreateBucketConfiguration={'LocationConstraint': region})
    print(f"\U0001f195 Created bucket '{bucket}'.")
    return bucket

ensure_bucket(CONTEXT_BUCKET, REGION)
s3.put_public_access_block(
    Bucket=CONTEXT_BUCKET,
    PublicAccessBlockConfiguration={
        'BlockPublicAcls': True, 'IgnorePublicAcls': True,
        'BlockPublicPolicy': True, 'RestrictPublicBuckets': True,
    },
)
print('\U0001f512 Public access blocked on bucket.')

## Step 2 — AgentCore execution role (create or reuse)

AgentCore Runtime assumes this role to run the agent. We start from the **official AgentCore Runtime execution role** in the [AWS docs](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-permissions.html) and add the two things **this** agent needs:

- **ECR** — pull the agent image (`BatchGetImage`, `GetDownloadUrlForLayer`, `GetAuthorizationToken`). *Missing these is the most common deploy failure.*
- **CloudWatch Logs + X-Ray + metrics** — observability.
- **Bedrock** — invoke the Claude model.
- **Workload identity tokens** — scoped to this agent.
- **AgentCore Memory data-plane** *(addition)* — the Strands session manager calls `ListEvents` / `CreateEvent` / `RetrieveMemoryRecords` to read and write the conversation. The official runtime policy omits these because it assumes an agent without conversation memory; without them, invocation fails at runtime with `AccessDenied` on `ListEvents`.
- **S3** *(addition)* — read/write the offload bucket so `ContextOffloader` can store and retrieve large tool outputs.

The **trust policy** uses `aws:SourceAccount` / `aws:SourceArn` conditions (confused-deputy protection), as the docs recommend.

In [ ]:
# Trust policy — only AgentCore, from this account, may assume the role (confused-deputy safe)
trust_policy = {
    'Version': '2012-10-17',
    'Statement': [{
        'Sid': 'AssumeRolePolicy',
        'Effect': 'Allow',
        'Principal': {'Service': 'bedrock-agentcore.amazonaws.com'},
        'Action': 'sts:AssumeRole',
        'Condition': {
            'StringEquals': {'aws:SourceAccount': account_id},
            'ArnLike': {'aws:SourceArn': f'arn:aws:bedrock-agentcore:{REGION}:{account_id}:*'},
        },
    }],
}

try:
    role = iam.get_role(RoleName=ROLE_NAME)
    iam.update_assume_role_policy(RoleName=ROLE_NAME, PolicyDocument=json.dumps(trust_policy))
    print(f"✅ Role '{ROLE_NAME}' already exists — reusing it (trust policy refreshed).")
except iam.exceptions.NoSuchEntityException:
    role = iam.create_role(
        RoleName=ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description='Execution role for the two-memory log-analysis AgentCore agent',
    )
    print(f"\U0001f195 Created role '{ROLE_NAME}'.")

ROLE_ARN = role['Role']['Arn']

# Official AgentCore Runtime execution policy (AWS docs), plus two additions for THIS agent:
#   - AgentCore Memory data-plane (the Strands session manager calls ListEvents/CreateEvent/Retrieve…)
#   - the S3 offload bucket
execution_policy = {
    'Version': '2012-10-17',
    'Statement': [
        {'Sid': 'ECRImageAccess', 'Effect': 'Allow',
         'Action': ['ecr:BatchGetImage', 'ecr:GetDownloadUrlForLayer'],
         'Resource': [f'arn:aws:ecr:{REGION}:{account_id}:repository/*']},
        {'Sid': 'ECRTokenAccess', 'Effect': 'Allow',
         'Action': ['ecr:GetAuthorizationToken'], 'Resource': '*'},
        {'Effect': 'Allow',
         'Action': ['logs:DescribeLogStreams', 'logs:CreateLogGroup'],
         'Resource': [f'arn:aws:logs:{REGION}:{account_id}:log-group:/aws/bedrock-agentcore/runtimes/*']},
        {'Effect': 'Allow', 'Action': ['logs:DescribeLogGroups'],
         'Resource': [f'arn:aws:logs:{REGION}:{account_id}:log-group:*']},
        {'Effect': 'Allow',
         'Action': ['logs:CreateLogStream', 'logs:PutLogEvents'],
         'Resource': [f'arn:aws:logs:{REGION}:{account_id}:log-group:/aws/bedrock-agentcore/runtimes/*:log-stream:*']},
        {'Effect': 'Allow',
         'Action': ['xray:PutTraceSegments', 'xray:PutTelemetryRecords',
                    'xray:GetSamplingRules', 'xray:GetSamplingTargets'],
         'Resource': ['*']},
        {'Effect': 'Allow', 'Action': 'cloudwatch:PutMetricData', 'Resource': '*',
         'Condition': {'StringEquals': {'cloudwatch:namespace': 'bedrock-agentcore'}}},
        {'Sid': 'GetAgentAccessToken', 'Effect': 'Allow',
         'Action': ['bedrock-agentcore:GetWorkloadAccessToken',
                    'bedrock-agentcore:GetWorkloadAccessTokenForJWT',
                    'bedrock-agentcore:GetWorkloadAccessTokenForUserId'],
         'Resource': [
             f'arn:aws:bedrock-agentcore:{REGION}:{account_id}:workload-identity-directory/default',
             f'arn:aws:bedrock-agentcore:{REGION}:{account_id}:workload-identity-directory/default/workload-identity/{AGENT_NAME}-*',
         ]},
        {'Sid': 'BedrockModelInvocation', 'Effect': 'Allow',
         'Action': ['bedrock:InvokeModel', 'bedrock:InvokeModelWithResponseStream'],
         'Resource': ['arn:aws:bedrock:*::foundation-model/*', f'arn:aws:bedrock:{REGION}:{account_id}:*']},
        # --- addition 1: AgentCore Memory data-plane (conversation memory read/write) ---
        # The Strands AgentCoreMemorySessionManager reads/writes conversation events. Without
        # these, invocation fails with AccessDenied on ListEvents (a 500 at runtime).
        {'Sid': 'AgentCoreMemoryDataPlane', 'Effect': 'Allow',
         'Action': ['bedrock-agentcore:CreateEvent', 'bedrock-agentcore:ListEvents',
                    'bedrock-agentcore:GetEvent', 'bedrock-agentcore:DeleteEvent',
                    'bedrock-agentcore:ListSessions', 'bedrock-agentcore:ListActors',
                    'bedrock-agentcore:RetrieveMemoryRecords',
                    'bedrock-agentcore:GetMemoryRecord', 'bedrock-agentcore:ListMemoryRecords'],
         'Resource': [f'arn:aws:bedrock-agentcore:{REGION}:{account_id}:memory/*']},
        # --- addition 2: the S3 offload bucket (ContextOffloader data memory) ---
        {'Sid': 'OffloadReadWrite', 'Effect': 'Allow',
         'Action': ['s3:GetObject', 's3:PutObject'],
         'Resource': f'arn:aws:s3:::{CONTEXT_BUCKET}/{CONTEXT_PREFIX}*'},
        {'Sid': 'OffloadList', 'Effect': 'Allow',
         'Action': ['s3:ListBucket'],
         'Resource': f'arn:aws:s3:::{CONTEXT_BUCKET}',
         'Condition': {'StringLike': {'s3:prefix': [f'{CONTEXT_PREFIX}*']}}},
    ],
}
iam.put_role_policy(
    RoleName=ROLE_NAME,
    PolicyName='AgentTwoMemoriesExecution',
    PolicyDocument=json.dumps(execution_policy),
)
print('✅ Execution policy attached (ECR + Logs + X-Ray + Bedrock + workload tokens + Memory data-plane + S3 offload).')
print(f'   ROLE_ARN = {ROLE_ARN}')

## Step 3 — AgentCore Memory (create if missing, reuse if it exists)

`MemoryClient.create_or_get_memory` is idempotent by name: it creates the memory resource with the given long-term strategies, or returns the existing one. We use session-summary and user-preference strategies, namespaced by `actorId`.

In [ ]:
from bedrock_agentcore.memory import MemoryClient

memory_client = MemoryClient(region_name=REGION)
memory = memory_client.create_or_get_memory(
    name=MEMORY_NAME,
    description='Conversation memory for the log-analysis incident agent',
    strategies=[
        {'summaryMemoryStrategy': {
            'name': 'SessionSummarizer',
            'namespaces': ['/summaries/{actorId}/{sessionId}/'],
        }},
        {'userPreferenceMemoryStrategy': {
            'name': 'PreferenceLearner',
            'namespaces': ['/preferences/{actorId}/'],
        }},
    ],
)
MEMORY_ID = memory.get('id')
print(f'\u2705 AgentCore Memory ready. MEMORY_ID = {MEMORY_ID}')

## Step 4 — Verify the S3 round-trip (exact-reference retrieval)

This is the behavior `ContextOffloader(S3Storage(...))` relies on: store bytes, get back an `s3://` reference, retrieve the same bytes verbatim. We clean up the probe object afterward.

In [ ]:
import asyncio, inspect, concurrent.futures
from strands.vended_plugins.context_offloader import S3Storage

def _run(maybe_coro):
    """Strands 1.44+ made Storage.store()/retrieve() async; 1.43 and earlier were sync.
    Also works inside Jupyter's running event loop, where asyncio.run() would raise."""
    if not inspect.isawaitable(maybe_coro):
        return maybe_coro
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(maybe_coro)
    with concurrent.futures.ThreadPoolExecutor(1) as ex:
        return ex.submit(asyncio.run, maybe_coro).result()

storage = S3Storage(bucket=CONTEXT_BUCKET, prefix=CONTEXT_PREFIX, region_name=REGION)
ref = _run(storage.store('probe', b'hello from setup notebook', 'text/plain'))
content, content_type = _run(storage.retrieve(ref))
assert content == b'hello from setup notebook', 'round-trip mismatch!'
print(f'✅ S3 round-trip OK — exact bytes returned by reference: {ref}')

s3.delete_object(Bucket=CONTEXT_BUCKET, Key=ref.replace(f's3://{CONTEXT_BUCKET}/', ''))
print('\U0001f9f9 Probe object deleted.')

## Step 5 — Deploy the agent to AgentCore Runtime (starter toolkit)

`Runtime().configure(...).launch(...)` packages `agentcore_production.py` + `agent_requirements.txt`, builds an ARM64 image **in the cloud with AWS CodeBuild** (no local Docker), pushes it to ECR, and creates the Runtime. We pass `memory_mode="STM_AND_LTM"` and the environment variables the agent reads (`BEDROCK_AGENTCORE_MEMORY_ID`, `CONTEXT_BUCKET`, `CONTEXT_PREFIX`).

**Deployment time:** ~3-5 minutes (CodeBuild builds the image + pushes to ECR).

In [ ]:
import os, glob
from bedrock_agentcore_starter_toolkit import Runtime

# Pre-flight cleanup: the toolkit caches the last deploy in .bedrock_agentcore.yaml.
# If that cached agent id was deleted/replaced, launch() tries to UPDATE a runtime
# that no longer exists and fails with ResourceNotFoundException. Removing the cached
# config makes configure() start clean and create (or cleanly reuse) the runtime.
for cfg in glob.glob('.bedrock_agentcore*.yaml'):
    os.remove(cfg)
    print(f'Removed stale config: {cfg}')

runtime = Runtime()
runtime.configure(
    entrypoint='agentcore_production.py',
    execution_role=ROLE_ARN,
    auto_create_ecr=True,
    requirements_file='agent_requirements.txt',
    region=REGION,
    agent_name=AGENT_NAME,
    memory_mode='STM_AND_LTM',
    deployment_type='container',
    non_interactive=True,
)

print('Launching (3-5 minutes)...')
result = runtime.launch(
    auto_update_on_conflict=True,
    env_vars={
        'AWS_REGION': REGION,
        'BEDROCK_AGENTCORE_MEMORY_ID': MEMORY_ID,
        'CONTEXT_BUCKET': CONTEXT_BUCKET,
        'CONTEXT_PREFIX': CONTEXT_PREFIX,
    },
)
AGENT_ARN = result.agent_arn
print(f'✅ Deployed. AGENT_ARN = {AGENT_ARN}')

## Step 6 — Invoke the deployed agent

The `actor_id` is sent as a custom HTTP header on the invoke request (the only way to scope long-term memory). Same `actor_id` across sessions → cross-session recall; large log outputs offload to S3 either way.

In [ ]:
import uuid, time
from botocore.exceptions import ClientError

def invoke_agent(agent_arn, prompt, session_id, actor_id, region=REGION, retries=4):
    """Invoke the Runtime, injecting the custom actor-id header via a boto3 event hook.

    Note 1: we create a FRESH client per call. The `before-sign` hook that adds the
    actor-id header must be registered on a clean client — reusing one client across
    calls can stack the hook and corrupt the SigV4 signature on the second request.

    Note 2: right after a deploy the runtime cold-starts and the first invoke can
    return a transient 500 (RuntimeClientError). We retry with backoff so the demo
    doesn't fail on the first call while the container warms up.
    """
    HEADER = 'X-Amzn-Bedrock-AgentCore-Runtime-Custom-Actor-Id'

    def add_header(request, **kwargs):
        request.headers.add_header(HEADER, actor_id)

    last_err = None
    for attempt in range(retries):
        client = boto3.client('bedrock-agentcore', region_name=region)
        client.meta.events.register_first('before-sign.bedrock-agentcore.InvokeAgentRuntime', add_header)
        try:
            resp = client.invoke_agent_runtime(
                agentRuntimeArn=agent_arn,
                runtimeSessionId=session_id,
                payload=json.dumps({'prompt': prompt}).encode(),
                qualifier='DEFAULT',
            )
            body = resp['response'].read().decode('utf-8')
            try:
                return json.loads(body).get('response', body)
            except json.JSONDecodeError:
                return body
        except ClientError as e:
            last_err = e
            code = e.response.get('Error', {}).get('Code', '')
            # Transient cold-start / 5xx — wait and retry. Permission/validation errors won't fix themselves.
            if 'RuntimeClientError' in code or '500' in str(e):
                wait = 5 * (attempt + 1)
                print(f'   cold start (attempt {attempt + 1}/{retries}), retrying in {wait}s...')
                time.sleep(wait)
                continue
            raise
    raise last_err

actor = f'sre-{str(uuid.uuid4())[:8]}'
sess = str(uuid.uuid4())

print('\U0001f464 Turn 1: fetch + analyze (logs offload to S3, conversation to AgentCore Memory)\n')
print(invoke_agent(AGENT_ARN,
    "Fetch 6 hours of logs for 'payment-service' and tell me which service had the most errors.",
    sess, actor))

print('\n\U0001f464 Turn 2: follow-up — same session, so the agent recalls Turn 1 from memory\n')
print(invoke_agent(AGENT_ARN,
    'Based on that, what should I investigate first?',
    sess, actor))

## Cleanup (optional)

Delete the resources this notebook created. Run only when you are done.

In [ ]:
# Full teardown — deletes EVERY resource this notebook created, so nothing is left
# billing or cluttering the account. Destructive: uncomment the call to run it.
import glob as _glob

def teardown():
    ctrl = boto3.client('bedrock-agentcore-control', region_name=REGION)
    ecr = boto3.client('ecr', region_name=REGION)
    cb = boto3.client('codebuild', region_name=REGION)

    # 1. AgentCore Runtime
    try:
        rid = AGENT_ARN.split('/')[-1]
        ctrl.delete_agent_runtime(agentRuntimeId=rid); print(f'🗑️  runtime {rid}')
    except Exception as e:
        print('runtime:', e)

    # 2. AgentCore Memory (both the one we created and the one the toolkit made for the agent)
    for mid in {MEMORY_ID} | {m['id'] for m in ctrl.list_memories().get('memories', [])
                              if 'LogAnalysis' in m['id'] or 'Incident' in m['id']}:
        try:
            ctrl.delete_memory(memoryId=mid); print(f'🗑️  memory {mid}')
        except Exception as e:
            print('memory', mid, e)

    # 3. ECR repository (force removes images)
    try:
        ecr.delete_repository(repositoryName=f'bedrock-agentcore-{AGENT_NAME.lower()}', force=True)
        print('🗑️  ECR repo')
    except Exception as e:
        print('ecr:', e)

    # 4. CodeBuild project
    try:
        cb.delete_project(name=f'bedrock-agentcore-{AGENT_NAME.lower()}-builder'); print('🗑️  CodeBuild project')
    except Exception as e:
        print('codebuild:', e)

    # 5. S3 bucket (empty then delete)
    try:
        objs = s3.list_objects_v2(Bucket=CONTEXT_BUCKET).get('Contents', [])
        for o in objs:
            s3.delete_object(Bucket=CONTEXT_BUCKET, Key=o['Key'])
        s3.delete_bucket(Bucket=CONTEXT_BUCKET); print(f'🗑️  bucket {CONTEXT_BUCKET}')
    except Exception as e:
        print('s3:', e)

    # 6. IAM role (inline policy first, then role)
    try:
        iam.delete_role_policy(RoleName=ROLE_NAME, PolicyName='AgentTwoMemoriesExecution')
        iam.delete_role(RoleName=ROLE_NAME); print(f'🗑️  role {ROLE_NAME}')
    except Exception as e:
        print('iam:', e)

    # 7. Local cached config so the next deploy starts clean
    for f in _glob.glob('.bedrock_agentcore*.yaml') + ['Dockerfile', '.dockerignore']:
        if os.path.exists(f):
            os.remove(f)
    print('🧹 local config cleaned')
    print('\n✅ Teardown complete — account is clean.')

# teardown()   # ← uncomment to delete everything this notebook created
print('Cleanup is defined but NOT run. Uncomment teardown() above to delete all resources.')

## Done

You now have a production agent with **two separate memories**:
- ✅ **Conversation** in AgentCore Memory (`STM_AND_LTM`, semantic recall, scoped by `actor_id`)
- ✅ **Context data** in S3 (`ContextOffloader(S3Storage(...))`, exact-reference retrieval)
- ✅ The execution role grants S3 read/write so the Runtime can offload tool data

The only change from the local demo (`test_native_pointer.py`) is the storage backend (`FileStorage` → `S3Storage`) and the model (`OpenAIModel` → `BedrockModel`, so no API key lives in the container).

### References

- [AgentCore Memory — Get started](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/memory-get-started.html)
- [AgentCore Runtime](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime.html)
- [Strands Context Management](https://strandsagents.com/docs/user-guide/concepts/context-management/)